day08

0. 복습

그래디언트 부스팅
 : 나무(결정나무)를 한 그루씩 순서대로 쌓되,
   앞의 오차를 다음 나무가 보정하는 앙상블
- 학습 데이터를 덜 외우면서 성능이 높다
- 하이퍼파라미터
	learning_rate(학습률) : 한 나무가 오차를 보정하는 정도
	n_estimators : 나무(단계) 수

1. 군집분석
 : 정답(타깃, 종속변수) 없이, 비슷한 데이터끼리 자동으로 무리(그룹)을
   지어 나누는 것
- 지금까지 처럼 "이건 생존, 저건 사망"이라고 미리 정답을 주는게 아니다
  데이터의 특성(독립변수)만 보고 "이것들은 서로 비슷하니 한 무리"라고
  스스로 묶는다(군집화)
- 각 무리를 군집(cluster)이라 하고, 이렇게 묶는 작업을 군집 분석이라 한다
ex) 쇼핑몰 고객 세분화
 : 쇼핑몰의 구매 기록만 보고 고객을 "알뜰족, 큰손, 신규"같은 그룹으로 나누는 것
   (행동이 비슷한 사람끼리 묶는다)
- 활용 : 고객 그룹 나누기, 비슷한 상품 묶기, 이상한 데이터 찾아내기(이상 탐지)

1) 비지도 학습
 : 지금까지 배운 모델은 모두 지도학습이지만
   군집분석은 비지도 학습이다
   핵심차이는 정답(타깃, 종속변수, y)의 유무이다

	지도 학습			비지도 학습
==============================================================================
정답(y) 있음(생존/사망, 품종 등)	없음(X만 사용)
기능	정답을 맞히도록 학습		비슷한 것끼리 스스로 묶기
모델	분류, 회귀			군집, 차원 축소
평가	정확도 등(정답과 비교)		정답이 없이 평가가 까다로움

** 비지도 학습 코드에는 y(정답, 타깃)가 없다. X(특성, 독립변수)만 모델에
   넣는다

2. K-means
- 대표를 정하고 모으기
- 데이터를 K개의 군집으로 나누는 가장 대표적인 방법
- K는 "몇 개로 나눌지", means(평균)는 "각 무리의 중심(평균 위치)"을
  뜻한다

1) K-means 작동방식
(1) 중심점 K개를 무작위로 뿌린다(각 무리의 임시대표)
(2) 각 데이터를 가장 가까운 중심점에 배정한다(가까운 대표에게 모임)
(3) 각 무리의 한가운데(평균 위치)로 중심점을 옮긴다
(4) 중심점이 더 이상 움직이지 않을 때까지 반복한다

2) k의 개수 (엘보우 메소드)
- k-means는 k(군집 개수)를 우리가 미리 정해줘야 한다
- 그런데 실제 데이터는 몇 개로 나눠야 좋은지 눈에 안보인다
- 이때 쓰는 것이 엘보우 메소드(방법)이다
  엘보우는 elbow 팔꿈치를 의미한다
- 기준은 inertia_(관성) 값이다. 각 점이 자기 중심점에서 얼마나 떨어져 있는지
  를 모두 더한 값으로, 작을수록 무리가 잘 뭉친 것이다
- K를 늘리면 inertia는 계속 줄지만(군집이 잘게 쪼개지니 당연)
  어느 순간부터 줄어드는 폭이 확 꺾인다
  그 꺾이는 지점(팔꿈치)이 적절한 k다.

3) 스케일링 중요성
- k-means는 거리(가까움)를 기준으로 묶는다
- 그래서 특성(X의 특성)마다 숫자 크기(스케일)가 제각각이면,
  큰 특성이 거리를 독차지해 엉뚱하게 군집화한다

## 군집분석(K-means)

In [ ]:
from sklearn.datasets import make_blobs

# make_blobs : 군집 연습용 가짜 데이터를 만들어 준다
# n_samples : 점 개수, centers = 덩어리 개수, cluster_std = 덩어리가 퍼진 정도
# 타깃(y)은 비지도 학습이라 존재 x => _로 받는다
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)

print(f"데이터 구조 : {X.shape}") # 점 300개, 각 점은(x, y) 2차원
print(X[:3])

# K-means 모델을 만들어 "4개로 군집화" 실행

In [ ]:
# 4개의 군집으로 군집화(y를 사용하지 않는다, 비지도 학습)
import numpy as np
from sklearn.cluster import KMeans # k-means모델

# 모델 생성
# n_clusters=k(몇개의 군집으로 나눌지), n_init = 여러번 시도해 죄선을 고름
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# fit_predict : 학습하면서 각 점이 몇번 군집인지 라벨을 돌려준다
# (X만 사용)
labels = kmeans.fit_predict(X)
print(f"각 점의 군집 라벨 : {labels[:15]}")
print(f"군집별 데이터 개수 : {[int(c) for c in np.bincount(labels)]}")
print(f"각 군집의 중심점 :\n {kmeans.cluster_centers_.round(2)}")

# 각 점(데이터)에 0~3번 군집 라벨이 붙었다
# 거의 고르게 4개의 무리(군집)로 나누었다
# cluster_centers_는 각 군집의 중심점 좌표다(그 무리(군집)의 대표위치)

In [ ]:
# 군집 시각화
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = "Malgun Gothic"
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=25, alpha=0.7)
# 중심점을 검은 X로 크게 표시
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=250,
           edgecolor='white', linewidths=1.5, label="군집 중심")

plt.title("K-means 군집 결과(K=4)")
plt.legend()
plt.show()
# 정답(타깃, y)을 전혀 주지 않았는데 4개의 군집을 정확하게 찾아냄
# 각 무리의 한가운데에 중심점(검은 X)이 자리 잡았다

### 적정 K(군집 개수) 값 찾기(엘보우 방법)

In [ ]:
# k를 1~7로 바꾸면서 inertia를 확인
inertias = []
for k in range(1, 8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertias.append(model.inertia_)
    print(f"k={k} | inertia={model.inertia_:.2f}")

In [ ]:
# 엘보우 메소드 시각화
plt.figure(figsize=(6, 4))
plt.plot(range(1, 8), inertias, "o-")
plt.xlabel("군집 개수 k")
plt.ylabel("inertia(군집 내 흩어짐)")
plt.title("엘보우 메소드")
plt.show()
# k=4까지는 급격하게 줄다가 그 뒤로부터는 완만해진다.
# 딱 팔꿈치처럼 꺾이는 k=4가 적절한 군집 개수다
# => 즉 "그래프가 꺾이는 지점"을 K로 고르면 된다

### 스케일링 중요성

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# 세 군집이 나오는 가짜 데이터(군집마다 데이터 개수는 60/60/60)
rng = np.random.RandomState(1)
# 특성 A(작은값), 특성 B(큰 값)

A = np.concatenate([rng.normal(0, 0.1, 60), rng.normal(1, 0.1, 60),
                   rng.normal(0.5, 0.1, 60)]) # 작은 스케일
B = np.concatenate([rng.normal(0, 30, 60), rng.normal(0, 30, 60),
                   rng.normal(400, 30, 60)]) # 큰 스케일

X_demo = np.column_stack([A, B])

# 스케일 전 vs 후 군집 결과 비교
labels_raw = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_demo)
labels_scaled = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(
        StandardScaler().fit_transform(X_demo) # 스케일링 데이터 특성    
)

print(f"스케일링 전 군집 크기 : {[int(c) for c in np.bincount(labels_raw)]}")
print(f"스케일링 후 군집 크기 : {[int(c) for c in np.bincount(labels_scaled)]}")

In [ ]:
# 스케일링 전 vs 후 군집 분석 결과 시각화
plt.figure(figsize=(11, 4))

ax1 = plt.subplot(1, 2, 1)
ax2 = plt.subplot(1, 2, 2)

# 왼쪽 - 스케일링 전 군집 결과
ax1.scatter(A, B, c=labels_raw, cmap="tab10", s=22, alpha=0.7)
ax1.set_title("스케일링 전")
ax1.set_xlabel("특성 A(작은 스케일)")
ax1.set_ylabel("특성 B(큰 스케일)")

# 오른쪽 - 스케일링 후 군집 결과
ax2.scatter(A, B, c=labels_scaled, cmap="tab10", s=22, alpha=0.7)
ax2.set_title("스케일링 후")
ax2.set_xlabel("특성 A(작은 스케일)")
ax2.set_ylabel("특성 B(큰 스케일)")

plt.show()
# 스케일링 전 : 큰 값인 특성 B가 거리를 지배해, 특성 A로만 갈리던 두 군집이
# 뒤섞였다(아래쪽 두 군집이 제대로 안 갈린게 보인다)
# 스케일링 후 : 두 특성을 공정하게 반영해 정확히 나뉜다
# 결론 : 거리 기반인 k-means는 스케일링을 반드시 해야한다

## 실제 데이터에 적용
> 붓꽃(iris)데이터의 품종 군집 분석, 군집한 뒤 결과가 실제 품종과 얼마나 맞는지 확인

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

# 붓꽃 3품종의 꽃잎, 꽃받침 길이/너비 4개 특성
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)

# 거리 기반이므로 스케일링 후 군집
X_scaled = StandardScaler().fit_transform(X)
clusters = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

# 실제 품종 이름(결과 비교용)
species = pd.Series(iris.target, name="실제품종").map(
    {0:"setosa", 1:"versicolor", 2:"virginica"}
)

# 실제 품종 vs 군집 결과로 교차표(crosstab)로 대조
ct = pd.crosstab(species, pd.Series(clusters, name="군집"))
ct
# 얼추 잘 군집화 되어 있는 것 같은데,
# 군집번호(0 , 1, 2)는 K-means가 아무렇게나 붙인 임의의 이름표라
# 실제 품종 순서와 안 맞기 때문에 헷갈린다

In [ ]:
# 각 군집을 "그 군집에서 가장 많은 품종"이름으로 바꿔 확인
# ct.idxmax() : 각 군집(열)에서 개수가 가장 많은 품종(행 이름)을 찾아준다
cluster_species = ct.idxmax()
print(cluster_species)

# 군집 번호를 대표 품종으로 이름으로 바꾸기
labels_named = pd.Series(clusters, name="군집->품종").map(cluster_species)

pd.crosstab(species, labels_named)
# 정답을 하나도 안 줬는데 군집이 실제 품종과 상당히 잘 맞는다
# - setosa는 50송이 전부 한 군집으로 완벽히 묶임
# - virginica, versicolor는 살짝 섞임

# K-means가 "이 붓꽃들을 대략 3종류로 나뉜다"는 구조를 스스로 발견

In [ ]:
# <군집 실습>
import seaborn as sns
# 펭귄으로 군집화(species 군집)
penguins = sns.load_dataset("penguins").dropna()
# 특성 : 부리 길이, 깊이, 날개 길이, 몸무게
features_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

X = penguins[features_cols]

In [ ]:
# 1) StandardScaler로 스케일링 후, k=3으로 K-means 군집하기
X_scaled = StandardScaler().fit_transform(X)
clusters = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

In [ ]:
# 2) 실제 종(penguins['species'])과 비교하여 군집 결과를 crosstab으로 비교
ct = pd.crosstab(penguins['species'], pd.Series(clusters, index=penguins.index, name="군집"))

# 각 군집에서 가장 많은 종
cluster_species = ct.idxmax()
labels_named = pd.Series(clusters, index=penguins.index, name="군집->종").map(cluster_species)

pd.crosstab(penguins['species'], labels_named)

### 과제

# 군집 분석과 K-means 과제

**정답을 알려주며 학습**시키는 지도학습이었다. 이번엔 **정답 없이 비슷한 것끼리 스스로 묶는**
비지도학습, **K-means 군집**이다. 그래서 이번 과제 코드에는 **정답 `y`가 등장하지 않는다.**
① 쇼핑몰 고객을 성향별로 묶어보고(문제 1), ② K-means에 **스케일링이 왜 필수인지** 확인한다(문제 2).

- 문제 1 데이터: **`day08_쇼핑고객.csv`**(고객 250명, 과제 노트북과 같은 폴더)
- 문제 2 데이터: scikit-learn 내장 **`load_wine`**(와인 178병)
- 각 문제는 `데이터 준비 → 스케일링 → 군집 → 결과 확인 → 해석` 흐름을 따른다.

## 문제 1) 쇼핑몰 고객 세분화 — 몇 개 그룹으로 나뉠까

쇼핑몰이 고객 250명의 **연간 구매액과 방문 횟수**만 가지고 있다. "알뜰족·큰손 같은 그룹이 있을 것 같은데
몇 개인지, 누가 어디 속하는지"는 아무도 모른다. **정답이 없는 상황** — 딱 군집이 필요한 자리다.

1. `day08_쇼핑고객.csv`를 불러와 **데이터 모양(`shape`)** 을 확인하고, 두 특성으로 **산점도**를 그려 눈으로 살펴본다.
2. K-means는 **거리 기반**이므로 `StandardScaler`로 **스케일링**한다.
   - 힌트: `X_scaled = StandardScaler().fit_transform(X)`
3. **엘보우 방법**으로 k를 정한다. k를 **1~7**로 바꿔가며 `inertia_`를 출력하고, **꺾은선 그래프**로 그린다.
4. 팔꿈치에서 찾은 k로 **군집**하고(`fit_predict`), **군집별 고객 수**(`np.bincount`)와 **중심점**(`cluster_centers_`)을 출력한다.
5. 1번의 산점도를 이번엔 **군집 라벨 색으로 칠해** 다시 그린다. (어떤 고객끼리 묶였는지 보기)
6. 결과를 **해석**한다. (엘보우가 어디서 꺾이나 / 몇 그룹으로 나뉘었나 / 각 군집은 어떤 성향의 고객인가)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.rcParams["font.family"] = "Malgun Gothic"  
plt.rcParams["axes.unicode_minus"] = False 

df = pd.read_csv("./day08_쇼핑고객.csv")
# df.head()
print(df.shape)

plt.figure(figsize=(7, 5))
plt.scatter(df['연간구매액'], df['방문횟수'], color="blue", alpha=0.6)
plt.title("쇼핑몰 고객 데이터")
plt.xlabel('연간구매액')
plt.ylabel("방문횟수")
plt.show()

X = df[['연간구매액', '방문횟수']]
X_scaled = StandardScaler().fit_transform(X)

inertias = []

for k in range(1, 8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)
    print(f"k={k} | inertia={model.inertia_:.2f}")
    
plt.figure(figsize=(7, 5))
plt.plot(range(1, 8), inertias, marker="o", color="red")
plt.title("엘보우")
plt.xlabel("군집 수")
plt.ylabel("inertia")
plt.show()

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
print(f"군집별 고객수 : {np.bincount(clusters)}")
print(f"군집별 중심점 : {kmeans.cluster_centers_}")

plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters, cmap="tab10", alpha=0.6)

centers = kmeans.cluster_centers_

plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=250,
           edgecolor='white', linewidths=1.5, label="군집 중심")

plt.title("K-means 군집 결과(K=4)")
plt.legend()
plt.show()

# 꺾이는 곳은 4
# 그룹은 4
# 그룹 1 연간 구매액, 방문 횟수 낮음/ 이탈 일반
# 그룹 2 연간구매액 낮음, 방문횟수 높음 / 알뜰
# 그룹 3 구매액 높음, 횟수 낮음 / 큰손
# 그룹 4 둘다 높음 / VIP 고객

## 문제 2) 스케일링이 군집을 좌우한다 — 와인 데이터

scikit-learn 내장 **와인 데이터**(178병, 특성 13개)로 군집해보자. 이 데이터는 특성마다 **숫자 크기가 극단적으로 다르다**
(어떤 건 0.3, 어떤 건 750). 마침 실제 품종 정답이 있으니, **스케일링 전과 후**의 군집이 실제 품종과 얼마나 맞는지 대조해보자.
(물론 **정답은 군집에 넣지 않는다.** 나중에 채점용으로만 쓴다.)

1. `load_wine`으로 데이터를 불러오고, **특성별 평균**을 출력해 크기 차이가 얼마나 큰지 확인한다.
2. **스케일링 없이** `k=3`으로 군집한 뒤, 실제 품종과 **`crosstab`으로 대조**한다.
   - 군집 번호는 임의로 붙으므로, 수업 6번처럼 **`idxmax()`로 각 군집의 대표 품종 이름을 붙여** 표를 정렬한다.
3. **`StandardScaler`로 스케일링한 뒤** 같은 방식으로 군집하고, 다시 **`crosstab`으로 대조**한다.
4. 두 표를 비교해 **어느 쪽이 실제 품종과 더 잘 맞는지**(대각선이 잘 맞는지) 확인한다.
5. 결과를 **해석**한다. (스케일링 전 표는 왜 엉망인가 / 어떤 특성이 거리를 독차지했나 /
   K-means에서 스케일링이 갖는 의미)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. 데이터 불러오기
# [wine 데이터] 이탈리아 같은 지역에서 재배된 세 품종의 와인 178병을 화학 분석한 결과.
#   - 특성 13개 : 알코올(alcohol), 마그네슘(magnesium), 색 강도(color_intensity),
#                 프롤린(proline, 아미노산의 일종) 등 화학 성분 측정값
#   - 실제 품종 : 0·1·2 세 가지 (군집에는 넣지 않고, 나중에 결과를 대조할 때만 쓴다)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)

print(X.mean())

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)
# clusters
# 가상의 품종 부여
species = pd.Series(wine.target, name="실제품종").map(
    {0: "품종1", 1: "품종2", 2: "품종3"}
)

ct = pd.crosstab(species, pd.Series(clusters, name="군집"))

cluster_species = ct.idxmax()
# print(cluster_species)

labels_named = pd.Series(clusters, name="군집 -> 품종").map(cluster_species)

print(pd.crosstab(species, labels_named))

scaled = StandardScaler()
X_scaled = scaled.fit_transform(X)
# X_scaled

kmeans_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_scaled = kmeans_scaled.fit_predict(X_scaled)
# clusters_scaled

ct_scaled = pd.crosstab(species, pd.Series(clusters_scaled, name="군집"))

cluster_scaled_species = ct_scaled.idxmax()

scaled_named = pd.Series(clusters_scaled, name="군집 -> 품종").map(cluster_scaled_species)

print(pd.crosstab(species, scaled_named))

# 특성마다 값의 크기가 다르게 때문에 큰 차이가 난다
# proline이 가장 커서 독차지함
# 스케일링을 해야 거리 계산을 지배하지 않고 비슷한 기준에서 거리가 반영된다